# Earnings Call Silence Mapping — Exploratory Analysis

**Strategy**: Transcript Topic Omission as an Alpha Signal  
**Approach**: Detect deliberate absences in prepared earnings remarks relative to company's historical language baseline.

---

This notebook walks through the full research pipeline end-to-end:
1. Universe & Data Coverage Overview
2. Text Preprocessing Demo
3. TF-IDF Matrix Construction
4. Standard Vocabulary Construction
5. Omission Detection Demo
6. Universe-Wide Omission Scoring
7. Hypothesis Testing
8. OLS Regression
9. Backtest Results
10. Limitations & Research Extensions

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import json
from pathlib import Path

import config

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('Environment ready. Project root:', config.PROJECT_ROOT)

---
## 1. Universe & Data Coverage Overview

Load the 20-company universe and the master corpus. Inspect coverage quality distribution — what fraction of 8-K filings contain full transcripts vs. press releases vs. insufficient content.

In [ ]:
# Load company universe
universe = pd.read_csv(config.COMPANY_UNIVERSE_PATH)
print(f'Universe: {len(universe)} companies')
display(universe)

In [ ]:
# Sector breakdown
fig, ax = plt.subplots(figsize=(10, 4))
universe['sector'].value_counts().plot(
    kind='barh', ax=ax, color=sns.color_palette('tab10', len(universe['sector'].unique()))
)
ax.set_title('Universe Sector Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Company Count')
plt.tight_layout()
plt.show()

In [ ]:
# Load universe corpus (if built)
if config.UNIVERSE_CORPUS_PATH.exists():
    corpus = pd.read_csv(config.UNIVERSE_CORPUS_PATH)
    print(f'Corpus loaded: {len(corpus)} rows, {corpus["ticker"].nunique()} companies')
    
    # Coverage quality breakdown
    cov_pct = corpus['coverage_quality'].value_counts(normalize=True).mul(100).round(1)
    print('\nCoverage quality breakdown:')
    display(cov_pct.to_frame('percentage'))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pie chart
    cov_counts = corpus['coverage_quality'].value_counts()
    axes[0].pie(
        cov_counts.values,
        labels=cov_counts.index,
        autopct='%1.1f%%',
        colors=['#2ca02c', '#ff7f0e', '#d62728'],
        startangle=90
    )
    axes[0].set_title('Coverage Quality Distribution', fontweight='bold')
    
    # Per-company transcript count
    transcript_counts = (
        corpus[corpus['coverage_quality'] == 'transcript']
        .groupby('ticker').size()
        .sort_values(ascending=False)
    )
    transcript_counts.plot(kind='bar', ax=axes[1], color='#1f77b4')
    axes[1].set_title('Full Transcripts per Company', fontweight='bold')
    axes[1].set_xlabel('Ticker')
    axes[1].set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Corpus not yet built. Run: build_universe_corpus(universe)')
    print('See README.md for full pipeline instructions.')

---
## 2. Text Preprocessing Demo

Show the raw vs cleaned transcript for one company-quarter, and demonstrate the prepared remarks / Q&A split.

In [ ]:
from src.text_processor import clean_transcript, split_transcript_sections, load_finance_stopwords

# Demo with synthetic text that mimics an earnings call structure
DEMO_TRANSCRIPT = """
UNITED STATES SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 8-K

Good morning, ladies and gentlemen, and thank you for joining TechCorp's Q3 2023 
earnings conference call. I'm John Smith, and with me today is our CFO, Jane Doe.

Before we begin, I would like to remind you that this call contains forward looking 
statements within the meaning of the safe harbor provisions of the Private Securities 
Litigation Reform Act. Actual results may differ materially.

In Q3, our cloud infrastructure segment delivered exceptional results with 34% 
year-over-year revenue growth. Our supply chain optimization initiatives reduced 
operating costs by 180 basis points. The artificial intelligence integration across 
our product portfolio continues to accelerate customer adoption and recurring revenue.

We remain confident in our full-year guidance of $12.4 billion in revenue.

QUESTION AND ANSWER SESSION
Operator: We will now begin the question-and-answer session. Your line is open.

Analyst: Can you give us more detail on gross margin trajectory?
CFO: Yes, going forward we expect basis points expansion from our cost reduction program.
"""

print('── RAW TEXT (first 500 chars) ──────────────────────────────────────────')
print(DEMO_TRANSCRIPT[:500])
print('\n── CLEANED TEXT (first 500 chars) ──────────────────────────────────────')
cleaned = clean_transcript(DEMO_TRANSCRIPT)
print(cleaned[:500])

In [ ]:
# Section split demo
sections = split_transcript_sections(DEMO_TRANSCRIPT)

print('── PREPARED REMARKS SECTION ────────────────────────────────────────────')
print(sections['prepared_remarks'][:600])
print('\n── Q&A SECTION ─────────────────────────────────────────────────────────')
print(sections['qa_section'][:400])

print(f'\nPrepared remarks length: {len(sections["prepared_remarks"])} chars')
print(f'Q&A length: {len(sections["qa_section"])} chars')

In [ ]:
# Stopwords demo
stopwords = load_finance_stopwords()
print(f'Total stopwords loaded: {len(stopwords)}')
print('\nSample custom financial stopwords:')
custom_samples = [sw for sw in sorted(stopwords) if len(sw.split()) > 1][:20]
for sw in custom_samples:
    print(f'  "{sw}"')

---
## 3. TF-IDF Matrix Construction

Show vocabulary statistics, most common bigrams/trigrams, and sample TF-IDF vectors.

In [ ]:
import pickle

if config.TFIDF_VECTORIZER_PATH.exists() and config.CORPUS_PREPROCESSED_PATH.exists():
    with open(config.TFIDF_VECTORIZER_PATH, 'rb') as f:
        vectorizer = pickle.load(f)
    
    corpus_pp = pd.read_csv(config.CORPUS_PREPROCESSED_PATH)
    feature_names = vectorizer.get_feature_names_out().tolist()
    
    print(f'Vocabulary size: {len(feature_names)} terms')
    print(f'N-gram range: {vectorizer.ngram_range}')
    print(f'Min document frequency: {vectorizer.min_df}')
    print(f'Sublinear TF: {vectorizer.sublinear_tf}')
    
    # Show feature distribution by n-gram length
    bigrams = [f for f in feature_names if len(f.split('_')) == 2 or len(f.split()) == 2]
    trigrams = [f for f in feature_names if len(f.split('_')) == 3 or len(f.split()) == 3]
    
    print(f'\nBigrams: {len(bigrams)}')
    print(f'Trigrams: {len(trigrams)}')
    
    # Top 20 terms by IDF (lowest IDF = most common across corpus)
    idf_series = pd.Series(vectorizer.idf_, index=feature_names).sort_values()
    print('\nMost universal terms (lowest IDF — common across all companies):')
    display(idf_series.head(20).to_frame('IDF weight'))
    
    print('\nMost distinctive terms (highest IDF — rare across corpus):')
    display(idf_series.tail(20).to_frame('IDF weight'))
else:
    print('TF-IDF matrix not yet built. Run the full pipeline first.')
    print('Example vocabulary terms this strategy targets:')
    example_terms = [
        'supply_chain_optimization', 'artificial_intelligence', 'gross_margin_expansion',
        'recurring_revenue_growth', 'operating_leverage', 'competitive_differentiation',
        'cloud_infrastructure', 'digital_transformation', 'free_cash_flow'
    ]
    for t in example_terms:
        print(f'  • {t}')

---
## 4. Standard Vocabulary Construction

Show the standard vocabulary for 2 companies — their top 20 terms, average TF-IDF weights, and core_vocab flags.

In [ ]:
if config.STANDARD_VOCABULARIES_PATH.exists():
    with open(config.STANDARD_VOCABULARIES_PATH, 'rb') as f:
        vocabularies = pickle.load(f)
    
    print(f'Standard vocabularies built for {len(vocabularies)} companies')
    
    # Show vocabulary for first 2 companies
    for i, (cik, cik_vocabs) in enumerate(vocabularies.items()):
        if i >= 2:
            break
        
        # Get most recent vocabulary
        latest_key = sorted(cik_vocabs.keys(), key=lambda k: (k[1], k[0]))[-1]
        vocab_df = cik_vocabs[latest_key]
        
        ticker = corpus_pp[corpus_pp['cik'].astype(str) == str(cik)]['ticker'].iloc[0] if 'corpus_pp' in dir() else cik
        print(f'\n── {ticker} Standard Vocabulary (as of {latest_key[0]}/{latest_key[1]}) ──')
        
        # Colour core vocab rows
        display(
            vocab_df[['rank', 'term', 'avg_tfidf_weight', 'quarters_present', 'core_vocab']]
            .style
            .apply(lambda x: ['background-color: #e6f3ff' if v else '' for v in x == x], axis=1)
            .format({'avg_tfidf_weight': '{:.4f}'})
        )
else:
    print('Standard vocabularies not yet built.')
    print('\nExample standard vocabulary structure:')
    example_vocab = pd.DataFrame({
        'rank': range(1, 11),
        'term': [
            'supply_chain_optimization', 'artificial_intelligence_integration',
            'cloud_revenue_growth', 'operating_margin_expansion',
            'free_cash_flow_generation', 'digital_transformation_initiatives',
            'customer_acquisition_cost', 'recurring_revenue_model',
            'competitive_differentiation', 'long_term_value_creation'
        ],
        'avg_tfidf_weight': [0.182, 0.171, 0.163, 0.151, 0.143, 0.138, 0.129, 0.121, 0.114, 0.108],
        'quarters_present': [4, 4, 3, 4, 4, 2, 4, 3, 4, 2],
        'core_vocab': [True, True, False, True, True, False, True, False, True, False]
    })
    display(example_vocab)

---
## 5. Omission Detection Demo

Run `detect_omissions` for one company's most recent quarter. Display the omission DataFrame and the topic presence heatmap.

In [ ]:
from src.omission_detector import detect_omissions, compute_omission_score

# Synthetic demo: create a fake current vector and standard vocab
# This shows how the detection logic works without requiring full pipeline data

np.random.seed(42)
N_TERMS = 20
N_FEATURES = config.MAX_FEATURES

# Synthetic standard vocabulary (top 20 terms for a hypothetical company)
demo_vocab_terms = [
    'supply_chain_optimization', 'artificial_intelligence', 'gross_margin_expansion',
    'cloud_revenue_growth', 'operating_leverage', 'competitive_differentiation',
    'free_cash_flow', 'digital_transformation', 'customer_retention_rate',
    'recurring_revenue', 'research_development_investment', 'market_share_expansion',
    'cost_reduction_initiative', 'product_innovation_pipeline', 'international_expansion',
    'partnership_ecosystem', 'data_analytics_platform', 'enterprise_customer_growth',
    'subscription_model_transition', 'operational_efficiency_improvement'
]

demo_vocab_df = pd.DataFrame({
    'term': demo_vocab_terms,
    'avg_tfidf_weight': np.random.uniform(0.05, 0.25, N_TERMS),
    'quarters_present': np.random.choice([2, 3, 4], N_TERMS, p=[0.2, 0.3, 0.5]),
    'core_vocab': None,
    'rank': range(1, N_TERMS + 1)
})
demo_vocab_df['core_vocab'] = demo_vocab_df['quarters_present'] == 4

# Synthetic feature names (universe vocabulary)
demo_feature_names = demo_vocab_terms + [f'other_term_{i}' for i in range(N_FEATURES - N_TERMS)]

# Current vector: some terms present, some omitted
# Omit supply_chain_optimization, gross_margin_expansion, operating_leverage
demo_current_vector = np.zeros(N_FEATURES)
for i, term in enumerate(demo_feature_names[:N_TERMS]):
    avg_w = demo_vocab_df.loc[demo_vocab_df['term'] == term, 'avg_tfidf_weight'].values[0]
    if term in ['supply_chain_optimization', 'gross_margin_expansion', 'operating_leverage',
                'free_cash_flow', 'digital_transformation']:
        # Simulate omission: present at only 5% of historical level
        demo_current_vector[i] = avg_w * 0.05
    else:
        # Present at ~90% of historical level
        demo_current_vector[i] = avg_w * np.random.uniform(0.7, 1.1)

# Run detection
omission_df = detect_omissions(demo_current_vector, demo_vocab_df, demo_feature_names)
score = compute_omission_score(omission_df)

print(f'Omission Score: {score:.4f}')
print(f'Omission Events: {omission_df["omission_event"].sum()} / {len(omission_df)} terms')
print(f'Core Vocab Omissions: {omission_df[omission_df["omission_event"] & omission_df["core_vocab"]].shape[0]}')

In [ ]:
# Display omission_df sorted by severity
display(
    omission_df
    .sort_values('omission_severity', ascending=False)
    .style
    .apply(lambda x: ['background-color: #ffe6e6' if v else '' for v in omission_df['omission_event']], axis=1)
    .format({
        'avg_tfidf_weight': '{:.4f}',
        'current_tfidf': '{:.4f}',
        'presence_ratio': '{:.3f}',
        'omission_severity': '{:.4f}'
    })
)

In [ ]:
# Visualise omission detection for this synthetic company
fig, ax = plt.subplots(figsize=(14, 7))

colors = [
    '#d62728' if row['omission_event'] else '#1f77b4'
    for _, row in omission_df.sort_values('omission_severity', ascending=True).iterrows()
]

sorted_df = omission_df.sort_values('omission_severity', ascending=True)
y_pos = range(len(sorted_df))
terms_display = [t.replace('_', ' ') for t in sorted_df['term']]

ax.barh(y_pos, sorted_df['avg_tfidf_weight'], color='#aec7e8', alpha=0.7, label='Historical avg')
ax.barh(y_pos, sorted_df['current_tfidf'], color=colors, alpha=0.9, label='Current quarter')

ax.set_yticks(y_pos)
ax.set_yticklabels(terms_display, fontsize=9)
ax.set_xlabel('TF-IDF Weight')
ax.set_title('Omission Detection: Current vs Historical TF-IDF Weights\n(Red = Omission Event)', 
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right')

# Mark omission threshold
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

---
## 6. Universe-Wide Omission Scoring

Distribution of omission scores across the universe. Top 5 highest and lowest omission companies with their top omitted terms.

In [ ]:
if config.OMISSION_SIGNALS_PATH.exists():
    signal_df = pd.read_csv(config.OMISSION_SIGNALS_PATH)
    
    print(f'Signal observations: {len(signal_df)}')
    print(f'Companies: {signal_df["ticker"].nunique()}')
    print(f'Date range: {signal_df[["year", "quarter"]].apply(lambda r: f"{r[\"quarter\"]}/{r[\"year\"]}", axis=1).min()} — '
          f'{signal_df[["year", "quarter"]].apply(lambda r: f"{r[\"quarter\"]}/{r[\"year\"]}", axis=1).max()}')
    
    print('\nOmission score distribution:')
    display(signal_df['omission_score'].describe().to_frame('omission_score').T)
    
    # Plot distribution
    from src.visualize import plot_omission_score_distribution
    from src.omission_detector import classify_omission_regime
    
    signal_df = classify_omission_regime(signal_df)
    plot_omission_score_distribution(signal_df)
    img = plt.imread(config.OUTPUTS_DIR / '01_omission_score_distribution.png')
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Signal file not yet generated.')
    
    # Show synthetic distribution for illustration
    np.random.seed(123)
    synthetic_scores = np.concatenate([
        np.random.exponential(0.02, 200),    # LOW_OMISSION
        np.random.normal(0.08, 0.03, 400),   # NEUTRAL
        np.random.gamma(3, 0.05, 200),       # HIGH_OMISSION
    ])
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(synthetic_scores, bins=40, color='#1f77b4', alpha=0.7, edgecolor='white')
    ax.set_title('Synthetic Omission Score Distribution (Illustrative)', fontweight='bold')
    ax.set_xlabel('Omission Score')
    ax.set_ylabel('Frequency')
    plt.show()

In [ ]:
# Top 5 highest and lowest omission companies
if config.OMISSION_SIGNALS_PATH.exists() and 'signal_df' in dir():
    print('TOP 5 HIGHEST OMISSION SCORES:')
    top5_high = signal_df.nlargest(5, 'omission_score')[[
        'ticker', 'quarter', 'year', 'omission_score', 
        'n_omission_events', 'n_core_vocab_omissions', 'top_omitted_terms'
    ]]
    for _, row in top5_high.iterrows():
        terms = json.loads(row['top_omitted_terms']) if pd.notna(row.get('top_omitted_terms')) else []
        print(f"  {row['ticker']} {row['quarter']}/{row['year']}: "
              f"score={row['omission_score']:.4f}, events={row['n_omission_events']}")
        print(f"    Top omitted: {', '.join(t.replace('_',' ') for t in terms[:3])}")
    
    print('\nTOP 5 LOWEST OMISSION SCORES:')
    top5_low = signal_df.nsmallest(5, 'omission_score')[[
        'ticker', 'quarter', 'year', 'omission_score',
        'n_omission_events', 'n_core_vocab_omissions'
    ]]
    display(top5_low)

---
## 7. Hypothesis Testing

Two-sample t-tests comparing HIGH_OMISSION vs LOW_OMISSION groups across three return horizons, followed by Spearman rank IC.

**Interpretation guide**: 
- A *negative* t-statistic means HIGH_OMISSION firms have lower mean returns than LOW_OMISSION firms — consistent with H₁.
- p-value < 0.05 (or < 0.017 with Bonferroni correction for 3 tests) indicates statistical significance.
- The 30-day horizon is the pre-specified primary test. 1-day and 5-day are exploratory.

In [ ]:
if config.OMISSION_SIGNALS_PATH.exists() and 'signal_df' in dir() and 'ret_30d' in signal_df.columns:
    from src.signal import run_ttest_validation, compute_information_coefficient
    
    ttest_results = run_ttest_validation(signal_df)
    ic_results = compute_information_coefficient(signal_df)
    
    # Visualise
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    horizons = ['ret_1d', 'ret_5d', 'ret_30d']
    labels = ['1-Day Return', '5-Day Return', '30-Day Return']
    
    for ax, col, label in zip(axes, horizons, labels):
        high = signal_df[signal_df['signal_label'] == 'HIGH_OMISSION'][col].dropna()
        low = signal_df[signal_df['signal_label'] == 'LOW_OMISSION'][col].dropna()
        
        ax.hist(high, bins=25, alpha=0.6, color='#d62728', label='HIGH_OMISSION', density=True)
        ax.hist(low, bins=25, alpha=0.6, color='#1f77b4', label='LOW_OMISSION', density=True)
        ax.axvline(high.mean(), color='#d62728', linestyle='--', linewidth=2)
        ax.axvline(low.mean(), color='#1f77b4', linestyle='--', linewidth=2)
        ax.set_title(label, fontweight='bold')
        ax.legend(fontsize=8)
        
        if col in ttest_results:
            res = ttest_results[col]
            ax.text(0.02, 0.97, f"p={res['p_value']:.3f}", transform=ax.transAxes,
                    fontsize=9, va='top')
    
    plt.suptitle('Return Distributions: HIGH_OMISSION vs LOW_OMISSION', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Return data not yet fetched. Run fetch_post_earnings_returns(signal_df) first.')

---
## 8. OLS Regression

Regress 30-day returns on the omission score and controls (sector FE, quarter FE).
HC3 robust standard errors control for heteroskedasticity.

**Key coefficient**: `omission_score` should be **negative and statistically significant** if H₁ holds.

In [ ]:
if config.OMISSION_SIGNALS_PATH.exists() and 'signal_df' in dir() and 'ret_30d' in signal_df.columns:
    from src.signal import run_ols_regression
    
    # Merge sector info from universe
    if 'sector' not in signal_df.columns:
        signal_df = signal_df.merge(
            universe[['ticker', 'sector']], on='ticker', how='left'
        )
    
    result = run_ols_regression(signal_df)
    
    if result is not None:
        # Extract key coefficients
        coeff_df = pd.DataFrame({
            'Coefficient': result.params,
            'Std Error (HC3)': result.bse,
            't-stat': result.tvalues,
            'p-value': result.pvalues
        })
        
        print('\nKey regression coefficients:')
        key_vars = [c for c in coeff_df.index if 'omission' in c.lower() or c == 'const']
        display(
            coeff_df.loc[key_vars]
            .style.format({
                'Coefficient': '{:.4f}',
                'Std Error (HC3)': '{:.4f}',
                't-stat': '{:.3f}',
                'p-value': '{:.4f}'
            })
            .apply(lambda x: [
                'background-color: #e6ffe6' if v < 0.05 else '' 
                for v in coeff_df.loc[key_vars, 'p-value']
            ], axis=1)
        )
else:
    print('Return data not yet available.')

---
## 9. Backtest Results

Long-short portfolio performance: LOW_OMISSION (long) vs HIGH_OMISSION (short), quarterly rebalancing.

In [ ]:
if 'signal_df' in dir() and 'ret_30d' in signal_df.columns:
    from src.backtest import run_long_short_backtest, run_sector_neutral_backtest, compute_performance_metrics
    from src.visualize import plot_cumulative_pnl, plot_return_by_decile
    
    # Run both variants
    bt_unconstrained = run_long_short_backtest(signal_df)
    bt_sector_neutral = run_sector_neutral_backtest(signal_df)
    
    print('UNCONSTRAINED LONG-SHORT:')
    metrics_unc = compute_performance_metrics(bt_unconstrained)
    
    print('\nSECTOR-NEUTRAL LONG-SHORT:')
    metrics_sn = compute_performance_metrics(bt_sector_neutral)
    
    # Comparison table
    comparison = pd.DataFrame({
        'Metric': ['Annualised Return', 'Annualised Sharpe', 'Max Drawdown', 'Win Rate', 'Calmar'],
        'Unconstrained': [
            f"{metrics_unc.get('annualized_return', 0):.2%}",
            f"{metrics_unc.get('annualized_sharpe', 0):.3f}",
            f"{metrics_unc.get('max_drawdown', 0):.2%}",
            f"{metrics_unc.get('win_rate', 0):.2%}",
            f"{metrics_unc.get('calmar_ratio', 0):.3f}",
        ],
        'Sector-Neutral': [
            f"{metrics_sn.get('annualized_return', 0):.2%}",
            f"{metrics_sn.get('annualized_sharpe', 0):.3f}",
            f"{metrics_sn.get('max_drawdown', 0):.2%}",
            f"{metrics_sn.get('win_rate', 0):.2%}",
            f"{metrics_sn.get('calmar_ratio', 0):.3f}",
        ]
    })
    print('\n── Strategy Comparison ─────────────────────────────────────────────────')
    display(comparison)
    
    # Return by decile chart
    plot_return_by_decile(signal_df)
    
    # Cumulative PnL chart
    plot_cumulative_pnl(bt_unconstrained, bt_sector_neutral)
    
    for chart in ['04_return_by_decile.png', '05_cumulative_pnl.png']:
        p = config.OUTPUTS_DIR / chart
        if p.exists():
            img = plt.imread(p)
            fig, ax = plt.subplots(figsize=(14, 6))
            ax.imshow(img)
            ax.axis('off')
            plt.show()
else:
    print('Backtest requires return data. Run fetch_post_earnings_returns() first.')

---
## 10. Limitations & Research Extensions

### Known Limitations

**1. EDGAR Transcript Coverage Gaps**  
Not all 8-K filings contain the full earnings call transcript. EDGAR is primarily a disclosure repository; companies are not required to file transcripts at all. Many file only the press release (exhibit 99.1) or a brief summary. This project uses a `coverage_quality` filter (>2000 words = "transcript") but the thresholds are heuristic and may misclassify lengthy press releases as transcripts. A production system would benefit from a dedicated transcript data vendor (e.g., Refinitiv, Bloomberg, FactSet).

**2. Section Detection Failure Rate**  
The prepared remarks / Q&A split relies on pattern matching heuristics. For filings without clear section headers, the full text is used for both sections. Log the failure rate — if >30% of documents fail to split, the Q&A contamination may meaningfully dilute the signal. This could be improved with a fine-tuned text classifier.

**3. Look-Ahead Bias in TF-IDF IDF Weights**  
The TF-IDF vectoriser is fitted on the *full* corpus, meaning IDF weights for a 2020 quarter partially reflect 2023 documents. This is a mild look-ahead bias in IDF weights (not in term presence). A production system would refit the vectoriser on a rolling window aligned to the signal date. The standard vocabulary trailing window itself does *not* have look-ahead bias.

**4. Survivorship Bias**  
The 20-company universe consists of current S&P 500 constituents — all survivors. Companies that experienced distress, were acquired, or were delisted are excluded. Survivorship bias tends to upwardly bias strategy returns. An extension should use a point-in-time universe.

**5. Multiple Comparison Problem**  
Three return horizons (1d, 5d, 30d) are tested simultaneously. Under H₀, each test has a 5% false positive rate. With 3 tests, the family-wise Type I error is ≈14%. Apply Bonferroni correction (α = 0.05/3 ≈ 0.017) when evaluating significance. The 30-day horizon is the pre-specified primary test.

---

### Research Extensions

1. **Apply to 10-Q MD&A sections**: The Management Discussion & Analysis in 10-Q filings is standardised, quarterly, and almost always includes the full text. This eliminates the EDGAR coverage gap entirely and provides a much larger, cleaner corpus.

2. **Dynamic omission threshold**: Rather than a fixed 20% threshold, fit the threshold to each company's historical distribution of presence ratios. A term that fluctuates 40% quarter-to-quarter needs a looser threshold than one that is rock-steady.

3. **Topic clustering with LDA**: Instead of treating each bigram/trigram independently, cluster vocabulary terms into latent topics (e.g., "supply chain", "competitive positioning", "margin outlook"). Omission of entire topic clusters is likely more informative than individual term drops.

4. **Analyst revision signals**: Cross the omission signal with post-earnings analyst estimate revisions. If high-omission companies show larger subsequent downward earnings revisions, this validates the channel of information asymmetry.

5. **Named entity omissions**: Detect when management stops mentioning specific product names, partnerships, or customer segments that were previously named. Entity-level omissions may carry stronger signal than generic bigrams.